# Push Gold Tables to AI Lakehouse
**Source:** `manufacturing_facility_ops.gold.*`  
**Target:** `ai_lakehouse.aidp.*`  
**Description:** Copies all agent-facing gold tables into the ai_lakehouse so AIDP agent tools can query them. Run this notebook after any gold layer refresh.

## 0. Setup

In [1]:
SOURCE_CATALOG = "manufacturing_facility_ops"
SOURCE_SCHEMA  = "gold"
TARGET_CATALOG = "ai_lakehouse"
TARGET_SCHEMA  = "aidp"

def push_table(table_name):
    source = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table_name}"
    target = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"
    df = spark.table(source)
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )
    count = spark.table(target).count()
    print(f"✅  {source}  →  {target}  —  {count} rows")

print(f"Source : {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target : {TARGET_CATALOG}.{TARGET_SCHEMA}")

Source : manufacturing_facility_ops.gold
Target : ai_lakehouse.aidp


## 1. Push defect_alert
**Agent:** Monitoring Agent  
Line-level defect rate breaches. Triggers the incident response workflow.

In [2]:
push_table("defect_alert")
spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.defect_alert").show(truncate=False)

✅  manufacturing_facility_ops.gold.defect_alert  →  ai_lakehouse.aidp.defect_alert  —  1 rows


+---------------+----------+--------------+---------------+---------------------+------------------+---------------+-------------+------------+---------------+------------+--------------------------+
|production_line|machine_id|total_ordered |total_completed|total_units_remaining|active_order_count|defect_rate_pct|threshold_pct|variance_pct|alert_triggered|alert_status|recorded_at               |
+---------------+----------+--------------+---------------+---------------------+------------------+---------------+-------------+------------+---------------+------------+--------------------------+
|Line-3         |M-102     |420.0000000000|122.0000000000 |298.0000000000       |2.0000000000      |8.3000000000   |5.0000000000 |3.3000000000|true           |BREACH      |2026-05-26 18:48:12.727638|
+---------------+----------+--------------+---------------+---------------------+------------------+---------------+-------------+------------+---------------+------------+--------------------------+


## 2. Push machine_risk_profile
**Agent:** ERP Context Agent + Maintenance Copilot Agent  
Full machine risk context — incident history, downtime, cost, risk level.

In [3]:
push_table("machine_risk_profile")
spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.machine_risk_profile") \
    .filter("machine_id = 'M-102'") \
    .show(truncate=False)

✅  manufacturing_facility_ops.gold.machine_risk_profile  →  ai_lakehouse.aidp.machine_risk_profile  —  8 rows


+----------+-------------+---------------+------------+---------------+-----------------+--------------------+------------------+--------------------------+-------------------+--------------------+-------------------+--------------+----------+
|machine_id|machine_name |production_line|status      |total_incidents|spindle_incidents|spindle_incidents_6m|total_downtime_hrs|total_maintenance_cost_usd|last_failure_date  |next_scheduled_maint|maintenance_overdue|risk_score    |risk_level|
+----------+-------------+---------------+------------+---------------+-----------------+--------------------+------------------+--------------------------+-------------------+--------------------+-------------------+--------------+----------+
|M-102     |CNC Lathe 102|Line-3         |Under Review|3.0000000000   |3.0000000000     |3.0000000000        |28.0000000000     |5630.0000000000           |2025-02-14 00:00:00|2025-02-09 00:00:00 |true               |150.0000000000|CRITICAL  |
+----------+------------

## 3. Push at_risk_orders
**Agent:** Production Impact Agent  
Active orders at risk due to machine issues with delivery urgency and revenue exposure.

In [4]:
push_table("at_risk_orders")
spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.at_risk_orders").show(truncate=False)

✅  manufacturing_facility_ops.gold.at_risk_orders  →  ai_lakehouse.aidp.at_risk_orders  —  2 rows


+--------+---------------+-------------+----------------+------------------+---------------+--------------+-------------------+---------------+--------+-------+---------------+----------+----------------+-------------------+
|order_id|product        |product_model|quantity_ordered|quantity_completed|units_remaining|completion_pct|due_date           |days_until_due |priority|status |production_line|machine_id|delivery_urgency|revenue_at_risk_usd|
+--------+---------------+-------------+----------------+------------------+---------------+--------------+-------------------+---------------+--------+-------+---------------+----------+----------------+-------------------+
|ORD-8812|Washing Machine|WM-8800      |240.0000000000  |88.0000000000     |152.0000000000 |36.7000000000 |2025-05-19 00:00:00|-372.0000000000|Critical|At Risk|Line-3         |M-102     |CRITICAL        |77520.0000000000   |
|ORD-8819|Washing Machine|WM-9000      |180.0000000000  |34.0000000000     |146.0000000000 |18.90000

## 4. Push spare_parts_availability
**Agent:** Supply Chain Agent  
Spare parts inventory for at-risk machines with availability status.

In [5]:
push_table("spare_parts_availability")
spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.spare_parts_availability").show(truncate=False)

✅  manufacturing_facility_ops.gold.spare_parts_availability  →  ai_lakehouse.aidp.spare_parts_availability  —  4 rows


+-----------+---------------------+---------------------+-----------+------------+-------------+---------------+--------------+-------------------+--------------------------------+
|part_number|part_name            |machine_compatibility|warehouse  |bin_location|qty_available|unit_cost_usd  |lead_time_days|availability_status|recommended_for_current_incident|
+-----------+---------------------+---------------------+-----------+------------+-------------+---------------+--------------+-------------------+--------------------------------+
|SP-4471    |CNC Spindle Assembly |M-101,M-102,M-105    |Warehouse B|B-12-04     |3.0000000000 |1850.0000000000|Same Day      |Available          |true                            |
|SP-4533    |Bearing Race Set     |M-101,M-102          |Warehouse B|B-12-07     |6.0000000000 |340.0000000000 |Same Day      |Available          |true                            |
|SP-4501    |Coolant Pump Impeller|M-101,M-108          |Warehouse B|B-08-06     |5.0000000000 

## 5. Push line_rerouting_options
**Agent:** Scheduling Agent  
Production lines capable of absorbing rerouted orders with capacity headroom.

In [6]:
push_table("line_rerouting_options")
spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.line_rerouting_options").show(truncate=False)

✅  manufacturing_facility_ops.gold.line_rerouting_options  →  ai_lakehouse.aidp.line_rerouting_options  —  2 rows


+-------+---------------+-----------------------+-----------+----------------------+----------------------+----------------------------+----------------+---------------+-----------------+
|line_id|line_name      |compatible_products    |status     |max_capacity_units_day|current_load_units_day|available_capacity_units_day|units_to_reroute|can_absorb_load|capacity_headroom|
+-------+---------------+-----------------------+-----------+----------------------+----------------------+----------------------------+----------------+---------------+-----------------+
|Line-5 |Assembly Line 5|WM-7700,WM-8800,WM-9000|Operational|500.0000000000        |120.0000000000        |380.0000000000              |298.0000000000  |true           |82.0000000000    |
|Line-1 |Assembly Line 1|WM-7700,WM-8800        |Operational|180.0000000000        |140.0000000000        |40.0000000000               |298.0000000000  |false          |-258.0000000000  |
+-------+---------------+-----------------------+-----------

## 6. Push machine_failure_scores (ML model output)
**Agent:** Maintenance Copilot Agent  
Failure probability scores per machine from the predictive maintenance model.
**Note:** Comment out this cell if the ML notebook has not finished running yet.

In [7]:
push_table("machine_failure_scores")
spark.table(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.machine_failure_scores") \
    .filter("machine_id = 'M-102'") \
    .show(truncate=False)

✅  manufacturing_facility_ops.gold.machine_failure_scores  →  ai_lakehouse.aidp.machine_failure_scores  —  8 rows


+----------+-------------+---------------+------------+----------+-------------------+-----------------------+---------------------+
|machine_id|machine_name |production_line|status      |risk_level|failure_probability|failure_probability_pct|model_recommendation |
+----------+-------------+---------------+------------+----------+-------------------+-----------------------+---------------------+
|M-102     |CNC Lathe 102|Line-3         |Under Review|CRITICAL  |1.0000000000       |100.0000000000         |IMMEDIATE MAINTENANCE|
+----------+-------------+---------------+------------+----------+-------------------+-----------------------+---------------------+



## 7. Final Validation

In [8]:
print(f"{'TABLE':<45} {'ROWS':>6} {'COLS':>6}")
print("-" * 60)
tables = [
    "defect_alert",
    "machine_risk_profile",
    "at_risk_orders",
    "spare_parts_availability",
    "line_rerouting_options",
]
for t in tables:
    full = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{t}"
    try:
        df = spark.table(full)
        print(f"{full:<45} {df.count():>6} {len(df.columns):>6}")
    except Exception as e:
        print(f"{full:<45} ERROR: {str(e)[:40]}")

print("\n✅  All agent tables available in ai_lakehouse.aidp")

TABLE                                           ROWS   COLS
------------------------------------------------------------


ai_lakehouse.aidp.defect_alert                     1     12


ai_lakehouse.aidp.machine_risk_profile             8     14


ai_lakehouse.aidp.at_risk_orders                   2     15


ai_lakehouse.aidp.spare_parts_availability         4     10


ai_lakehouse.aidp.line_rerouting_options           2     10

✅  All agent tables available in ai_lakehouse.aidp
